# Chapter 6. Ensemble Learning and Random Forests

The fundamental philosophy of ensemble learning is the wisdom of the crowd. Aggregating the predictions of a group of diverse models often yields significantly higher accuracy and robustness than relying on the single best individual predictor.

## Voting Classifiers

A voting classifier aggregates the predictions of multiple diverse baseline models. To maximize effectiveness the baseline models should be structurally independent and utilize completely different algorithms. This ensures they make uncorrelated mathematical errors.

* **Hard Voting:** The ensemble strictly predicts the class that receives the absolute majority of votes.
* **Soft Voting:** The ensemble predicts the class with the highest average probability across all constituent models. Soft voting mathematically grants more weight to highly confident predictions and often outperforms hard voting.

**Probability Prerequisites:**
Soft voting strictly requires every single classifier in the ensemble to be capable of estimating class probabilities. Algorithms that do not output probabilities by default such as Support Vector Machines must be explicitly configured to do so before joining a soft voting ensemble.

The mathematical engine driving this is the Law of Large Numbers. Even if individual models are weak learners barely exceeding 51% accuracy an ensemble of 1000 independent models can mathematically push the aggregated accuracy closer to 75%.

First let's construct a Hard Voting ensemble to establish a performance baseline. We will evaluate the individual models and observe how the majority vote mechanism operates.

In [17]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# Generate and split a nonlinear toy dataset
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Train Hard Voting Classifier
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(random_state=42))
    ]
)
voting_clf.fit(X_train, y_train)

# Evaluate individual baseline models
for name, clf in voting_clf.named_estimators_.items():
    print(name, '=', clf.score(X_test, y_test))

lr = 0.864
rf = 0.896
svc = 0.896


In [23]:
# Show hard voting mechanism for the first instance
print('Ensemble Prediction \n', voting_clf.predict(X_test[:1]))
print('Individual Predictions \n', [clf.predict(X_test[:1]) for clf in voting_clf.estimators_])

Ensemble Prediction 
 [1]
Individual Predictions 
 [array([1], dtype=int64), array([1], dtype=int64), array([0], dtype=int64)]


In [25]:
# Evaluate Hard Voting Ensemble
print('Hard Voting Accuracy \n', voting_clf.score(X_test, y_test))

Hard Voting Accuracy 
 0.912


The voting classifier predicts class 1 for the first instance of the test set, because two out of three classifiers predict that class.

Now let's try Soft Voting to see if using probability estimates improves our accuracy. Since Soft Voting relies on confidence levels rather than just a strict majority count we first need to explicitly enable probability calculations for our SVM classifier.

In [35]:
# Upgrade to Soft Voting dynamically
voting_clf.voting = 'soft'
voting_clf.named_estimators['svc'].probability = True
voting_clf.fit(X_train, y_train)

# Evaluate Soft Voting Ensemble
print('Soft Voting Accuracy \n', voting_clf.score(X_test, y_test))

Soft Voting Accuracy 
 0.92
